# Preprocesado de datos y división del dataset

En este notebook vamos a preparar los datos para el entrenamiento del modelo clasificador de riesgo IA (AI Act).

Los pasos que seguiremos son:
1. Carga del dataset limpio
2. Aplicar NER (Named Entity Recognition) a la columna `descripcion`
3. Explorar las entidades extraídas por tipo y clase de riesgo
4. Preprocesar el texto con lematización
5. Dividir en train / validation / test (70% / 15% / 15%) con estratificación
6. Guardar los conjuntos en `data/processed/`

In [10]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_fusionado"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_dataset_fusionado"
functions._DATASET_TAGS = {"dataset_type": "fusionado", "dataset_source": "eu_ai_act_flagged"}

In [11]:
#Al principio de cada notebook, añadimos estas líneas para que se recarguen automáticamente las funciones que hayamos editado en el módulo functions.py sin tener que reiniciar el kernel cada vez.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import pandas as pd

path = "datasets/eu_ai_act_flagged_es_limpio.csv"
df = pd.read_csv(path)

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")
print("\nDistribución de clases:")
print(df["etiqueta"].value_counts())

Dataset cargado: 600 filas, 17 columnas
Columnas: ['ambiguity', 'articles', 'category', 'context', 'descripcion', 'etiqueta', 'explanation', 'id', 'notas', 'sector', 'severity', 'split', 'tipo_datos', 'violation', 'etiqueta_normalizada', 'longitud', 'descripcion_limpia']

Distribución de clases:
etiqueta
inaceptable        208
riesgo_minimo      169
alto_riesgo        136
riesgo_limitado     87
Name: count, dtype: int64


## 1. Aplicar NER al dataset

In [13]:
from functions import extraer_entidades, resumen_entidades

# Aplicamos NER a la columna de descripción original (sin limpiar)
df = extraer_entidades(df, "descripcion")

# Mostramos las entidades de las primeras 3 filas como ejemplo
for i in range(3):
    print(f"Texto: {df['descripcion'].iloc[i][:100]}...")
    print(f"Entidades: {df['entidades'].iloc[i]}")
    print()

Texto: Una ciudad inteligente implementa un sistema de inteligencia artificial para la optimización del sem...
Entidades: []

Texto: Un chatbot de servicio al cliente utiliza el reconocimiento emocional para detectar la frustración d...
Entidades: []

Texto: Un sistema de inteligencia artificial utilizado por una compañía de seguros para evaluar los factore...
Entidades: []



In [14]:
# Resumen de entidades por tipo y por clase de riesgo
freq_por_tipo, freq_por_tipo_clase = resumen_entidades(df)

Frecuencia global de entidades por tipo:
tipo_entidad
MISC    93
LOC     90
PER     25
ORG      8
Name: count, dtype: int64

Frecuencia de entidades por tipo y clase de riesgo:
tipo_entidad     LOC  MISC  ORG  PER
clase                               
alto_riesgo       15    24    4    8
inaceptable       23    19    0    2
riesgo_limitado   18     5    2   12
riesgo_minimo     34    45    2    3


### Conclusión del NER

Las entidades nombradas (NER) en este dataset regulatorio no aportan señal discriminativa significativa para la clasificación de riesgo. Los textos describen sistemas genéricos de IA sin mencionar nombres propios, organizaciones o ubicaciones específicas que diferencien las clases.

Por tanto, **continuamos el entrenamiento utilizando solo el texto limpio y lematizado** como feature principal.

## 2. Preprocesado del texto con lematización

In [6]:
from functions import preparar_dataset

# Features seguras a añadir junto al texto lematizado.
# NUNCA incluir: violation, severity (leakage directo), ambiguity (leakage minoritaria),
# explanation (no disponible en inferencia), split (metadato del pipeline).
EXTRA_FEATURES = ["category", "context", "longitud", "num_articles"]

df_processed = preparar_dataset(df, "descripcion", "etiqueta", extra_columns=EXTRA_FEATURES)

print(f"Dataset procesado: {df_processed.shape}")
print(f"Columnas: {list(df_processed.columns)}")

print("\nEjemplo de texto lematizado:")
print(f"  Original:   {df['descripcion'].iloc[0][:100]}...")
print(f"  Lematizado: {df_processed['text_final'].iloc[0][:100]}...")

print("\nEjemplo de features estructuradas:")
print(df_processed[["category", "context", "longitud", "num_articles"]].head())

Dataset procesado: (600, 6)
Columnas: ['text_final', 'etiqueta', 'category', 'context', 'longitud', 'num_articles']

Ejemplo de texto lematizado:
  Original:   Una ciudad inteligente implementa un sistema de inteligencia artificial para la optimización del sem...
  Lematizado: ciudad inteligente implementar sistema inteligencia artificial optimización semáforo incluir panel p...

Ejemplo de features estructuradas:
              category               context  longitud  num_articles
0         transparency    ciudad inteligente       346             2
1      risk_management   servicio al cliente       316             3
2  accuracy_robustness                seguro       291             1
3      data_governance  comercio electrónico       321             1
4    high_risk_systems    ciudad inteligente       405             2


## 3. División en train / validation / test

In [7]:
from functions import split_dataset

# Dividimos en train (70%), validation (15%) y test (15%) con estratificación
train_df, val_df, test_df = split_dataset(df_processed, "etiqueta")

print("\nDistribución por clase en cada conjunto:")
print("\nTrain:")
print(train_df["etiqueta"].value_counts())
print("\nValidation:")
print(val_df["etiqueta"].value_counts())
print("\nTest:")
print(test_df["etiqueta"].value_counts())

Train: 420 muestras
Validation: 90 muestras
Test: 90 muestras

Distribución por clase en cada conjunto:

Train:
etiqueta
inaceptable        146
riesgo_minimo      118
alto_riesgo         95
riesgo_limitado     61
Name: count, dtype: int64

Validation:
etiqueta
inaceptable        31
riesgo_minimo      26
alto_riesgo        20
riesgo_limitado    13
Name: count, dtype: int64

Test:
etiqueta
inaceptable        31
riesgo_minimo      25
alto_riesgo        21
riesgo_limitado    13
Name: count, dtype: int64


## 4. Guardar los conjuntos procesados

In [8]:
import os

output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(output_dir, "validation.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

print(f"Archivos guardados en {output_dir}/:")
for f in os.listdir(output_dir):
    print(f"  {f}")

Archivos guardados en data/processed/:
  test.csv
  train.csv
  validation.csv


In [9]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
from functions import log_mlflow_safe

try:
    log_mlflow_safe(
        run_name="preprocesado",
        params={
            "lemmatize":      True,
            "ner":            True,
            "test_size":      0.15,
            "val_size":       0.15,
            "random_state":   42,
            "extra_features": "category,context,longitud,num_articles",
        },
        metrics={
            "n_train":       len(train_df),
            "n_val":         len(val_df),
            "n_test":        len(test_df),
            "n_features_out": df_processed.shape[1] - 1,  # excluye etiqueta
        },
    )
    print("✓ Preprocesado registrado en MLflow")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://34.244.146.100/mlflow
⚠ MLflow no disponible: API request to https://34.244.146.100/mlflow/api/2.0/mlflow/experiments/get-by-name failed with timeout exception HTTPSConnectionPool(host='34.244.146.100', port=443): Max retries exceeded with url: /mlflow/api/2.0/mlflow/experiments/get-by-name?experiment_name=clasificador_riesgo_dataset_fusionado (Caused by ConnectTimeoutError(<HTTPSConnection(host='34.244.146.100', port=443) at 0x2592404e630>, 'Connection to 34.244.146.100 timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120, type: int) to a larger value.
